# Module 7 - Lab 2: Build a Vector Store and Retriever with LangChain and FAISS

**Module:** 7 - Basic LLM Systems (RAG) | **Scenario:** IT Operations Knowledge Assistant | **Continues from:** Module 7 Lab 1 (Simple RAG)

Lab 1 showed the complete RAG flow in one notebook. This lab goes one level deeper into the **vector store** and the **retriever**. You will build a FAISS index, save it to disk, reload it later **without generating the document embeddings again**, and use a retriever to answer IT Operations questions.

**Learning objectives**

1. Explain the role of a vector database in RAG.
2. Generate embeddings from operations records.
3. Create a local FAISS vector store that keeps metadata.
4. Persist the FAISS index to a local folder.
5. Reload a saved index and check whether it still matches the source data.
6. Create a LangChain retriever and run similarity searches with it.
7. Use the retrieved records to answer IT Operations questions with an LLM.

**How to run this notebook:** execute the cells **in order from top to bottom**. Open the notebook from the project folder and make sure the `.env` file is in place (see the README).

## Section 1 - Introduction to Vector Stores

Lab 1 introduced RAG. This section focuses on the part that stores and searches the knowledge base.

**What is a vector store?** A vector store is a database for embeddings. It keeps each text chunk together with its vector, and it can quickly find the chunks whose vectors are closest to a query vector.

**What is an embedding?** An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar meaning get vectors that point in similar directions, even when they use different words.

**What is semantic similarity?** It is a measure of how close two embeddings are. The question "high processor usage" is close to the record "Server CPU utilization above 90 percent", although the two texts share no keywords.

**What is FAISS?** FAISS (Facebook AI Similarity Search) is an open-source library for fast similarity search over vectors. In this lab it runs locally inside Python. It needs no server, no account and no external database. Its index can be saved to a folder and loaded again later.

**What is a retriever?** A retriever is a LangChain component with one job: given a question, return the most relevant documents. Most retrievers are a thin wrapper around a vector store.

**Vector store or retriever?**

| | Vector store | Retriever |
|---|---|---|
| What it is | The index that stores and searches vectors | A simple interface: question in, documents out |
| Typical methods | `from_documents`, `save_local`, `load_local`, `similarity_search` | `invoke(question)` |
| Who uses it | The application, when it builds and saves the index | The RAG chain, when it needs context |

**Traditional search vs. semantic search**

| | Traditional keyword search | Semantic similarity search |
|---|---|---|
| How it matches | Finds the same words | Finds the same meaning |
| Query "high processor usage" | Misses "CPU utilization" because the words differ | Finds "Server CPU utilization above 90 percent" |
| Handles synonyms | No | Yes |

**Why persistence matters.** Creating an index means sending every chunk to the embedding model. That takes time and costs money. Saving the index to disk lets the application reuse it the next time it runs.

**Workflow used in this lab:**

```mermaid
flowchart LR
    A[Documents] --> B[Embeddings]
    B --> C[FAISS Index]
    C --> D[Persistent Storage]
    D --> E[Retriever]
    E --> F[Relevant Documents]
```

## Section 2 - Environment Setup

This lab uses the **Python interpreter already configured in VS Code**. Do not create a new virtual environment, install another Python or change the default interpreter. The notebook must run on the same interpreter where `python -m pip install -r requirements.txt` was run.

**Packages.** The LangChain packages are pinned to the same versions as Module 7 Lab 1: `langchain`, `langchain-core`, `langchain-community`, `langchain-openai` and `langchain-text-splitters`. `faiss-cpu` provides the FAISS index. `python-dotenv` reads the API key from the `.env` file.

**What this section checks:** the Python version, the installed LangChain versions, and whether the OpenAI API key is available. The key itself is never printed.

In [ ]:
# Standard library imports used in this notebook
import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

# LangChain, FAISS and OpenAI imports
from dotenv import load_dotenv
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All libraries imported.\n")

# Show the Python version and the installed package versions
print("Python:", sys.version.split()[0])
for package in ["langchain", "langchain-core", "langchain-community",
                "langchain-openai", "langchain-text-splitters", "faiss-cpu", "python-dotenv"]:
    print(f"{package:26} {version(package)}")

# Read OPENAI_API_KEY from the .env file in this folder (the key is never written in the notebook)
load_dotenv()

# Check that the key is available. Only True/False is printed, never the key itself.
api_key_found = bool(os.getenv("OPENAI_API_KEY"))
print("\nOPENAI_API_KEY loaded:", api_key_found)
if not api_key_found:
    raise EnvironmentError("OPENAI_API_KEY not found. Copy .env.example to .env and add your key.")

In [ ]:
# ---- Lab settings: change these values to experiment ----
CHAT_MODEL = "gpt-4o-mini"                  # the LLM that writes the answer
EMBEDDING_MODEL = "text-embedding-3-small"  # the model that turns text into vectors
KB_PATH = "operations_knowledge_base.csv"   # the IT Operations knowledge base (CSV)
INDEX_DIR = "faiss_operations_index"        # folder where the FAISS index is saved
CHUNK_SIZE = 500                            # maximum characters per chunk
CHUNK_OVERLAP = 50                          # characters shared by neighbouring chunks
TOP_K = 3                                   # number of chunks returned per search
REBUILD_INDEX = False                       # True = always rebuild the index on this run

# ---- Model objects (creating them does not call the API yet) ----
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)   # text -> vector
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)      # temperature=0 keeps answers consistent

print("Embedding model:", EMBEDDING_MODEL)
print("Chat model:     ", CHAT_MODEL)
print("Index folder:   ", INDEX_DIR)
print("REBUILD_INDEX:  ", REBUILD_INDEX)

## Section 3 - Load the Operations Knowledge Base

This is the same `operations_knowledge_base.csv` used in Module 7 Lab 1. It has **29 records**, one per row, with the columns `Record_ID`, `Category`, `Title`, `Description`, `Procedure`, `Severity` and `Escalation`.

**What is a Document?** A LangChain `Document` has two parts:

- `page_content`: the text that the embedding model will read.
- `metadata`: information about the source, such as the file name and the row number.

**What CSVLoader does:** it creates one `Document` per CSV row. Each column is written as `Column: value` on its own line, so the record keeps its structure.

In [ ]:
# Stop early with a clear message if the knowledge base is not in this folder
if not Path(KB_PATH).exists():
    raise FileNotFoundError(f"{KB_PATH} not found. Run the notebook from the project folder.")

# CSVLoader creates one Document per row. utf-8-sig also reads files saved by Excel.
loader = CSVLoader(file_path=KB_PATH, encoding="utf-8-sig")
documents = loader.load()

print("Number of documents loaded:", len(documents))

# Sample document: the text (page_content) and the source information (metadata)
print("\n--- Sample document: page_content ---")
print(documents[0].page_content)

print("\n--- Sample document: metadata ---")
print(documents[0].metadata)

## Section 4 - Text Splitting

**Why split documents?** One embedding represents one piece of meaning. A long text with several topics produces a blurred vector. Splitting gives each vector a focused passage, so a search can return the passage that answers the question.

**Key terms**

- **Chunk size:** the maximum number of characters in one chunk. This lab uses `CHUNK_SIZE = 500`. Small chunks match a question precisely but can lose context. Large chunks keep context but blur the match.
- **Chunk overlap:** characters repeated between neighbouring chunks. This lab uses `CHUNK_OVERLAP = 50`. It stops a sentence cut at a boundary from being lost completely.

`RecursiveCharacterTextSplitter` tries to split at paragraph breaks first, then at line breaks, then at spaces. Each CSV field is on its own line, so most splits happen between fields. Every chunk keeps the metadata of its source row. Continuation chunks may not repeat the `Record_ID`, so the row number in the metadata is what links them back to the record.

In [ ]:
# Configure the splitter with the chunk settings from the configuration cell
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Split every document into chunks. Metadata is copied to each chunk.
chunks = text_splitter.split_documents(documents)

print(f"{len(documents)} documents were split into {len(chunks)} chunks.")

# Show two sample chunks with their metadata
for index in range(2):
    print(f"\n--- Chunk {index} | metadata: {chunks[index].metadata} ---")
    print(chunks[index].page_content)

## Section 5 - Generate Embeddings

**What is an embedding?** The embedding model reads a piece of text and returns a list of numbers (a vector). The model `text-embedding-3-small` returns 1,536 numbers for every text. No single number has a human-readable meaning. Together they place the text at a point in a "meaning space".

**Why similar meanings give similar vectors.** The model is trained so that texts about the same topic end up close together, even when they use different words. "High CPU utilization on an application server" and "Server CPU utilization above 90 percent" end up close to each other.

**Why embeddings cost money.** Every chunk sent to the embedding model is billed by OpenAI. The cost is small for this lab, but it grows with the size of the knowledge base. Every question is embedded too, so every search has a small cost. Section 13 explains how persistence reduces this cost.

The `embeddings` object was created in Section 2. This cell uses it once to show what an embedding looks like.

In [ ]:
sample_text = "High CPU utilization on an application server"
sample_vector = embeddings.embed_query(sample_text)   # one embedding call

print("Sample text:", sample_text)
print("Embedding dimensions:", len(sample_vector))
print("First 5 values:", [round(value, 4) for value in sample_vector[:5]])

## Section 6 - Create the FAISS Vector Store

**What is FAISS?** FAISS is a library for similarity search. It stores vectors in an *index* and finds the vectors closest to a query very quickly. It suits a classroom or a small knowledge base because it runs inside Python.

**What gets stored?** `FAISS.from_documents()` does three things:

1. Sends every chunk's text to the embedding model and receives a vector back.
2. Stores the vectors in the FAISS index.
3. Keeps each chunk's text and metadata in a document store, linked to its vector.

**How similarity search works.** The question is embedded with the same model. FAISS compares the question vector with every chunk vector and returns the `k` closest chunks, together with their text and metadata. A FAISS *score* is a distance, so a **lower** score means a closer match.

**Build or reuse?** Building the index means embedding every chunk, which is the slow and costly step. The cell below checks the saved index first. The three possible states are:

- **missing:** no saved index was found, so the index is built.
- **changed:** the saved index was built from a different CSV, chunk settings or embedding model, so it is rebuilt.
- **current:** the saved index matches the data, so it is loaded from disk and **no document embeddings are generated**.

`REBUILD_INDEX = True` in the configuration cell forces a rebuild. Section 9 explains the full rule.

**Security note:** the cell below loads an index with `allow_dangerous_deserialization=True`. This is safe only for an index that this notebook created on your own machine. Section 8 explains why.

In [ ]:
def file_sha256(file_path):
    """Return the SHA-256 checksum of a file. Any change to the file changes the checksum."""
    return hashlib.sha256(Path(file_path).read_bytes()).hexdigest()


# The settings below decide whether a saved index is still valid
FINGERPRINT_KEYS = ["source_sha256", "embedding_model", "chunk_size", "chunk_overlap"]


def make_fingerprint():
    """Describe the current source file and settings."""
    return {
        "source_sha256": file_sha256(KB_PATH),
        "embedding_model": EMBEDDING_MODEL,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
    }


def check_index(index_dir, fingerprint):
    """Return 'missing', 'changed' or 'current' for the saved index in index_dir."""
    index_dir = Path(index_dir)
    metadata_path = index_dir / "index_metadata.json"
    required_files = [index_dir / "index.faiss", index_dir / "index.pkl", metadata_path]
    if not all(path.exists() for path in required_files):
        return "missing"

    saved = json.loads(metadata_path.read_text(encoding="utf-8"))
    for key in FINGERPRINT_KEYS:
        if saved.get(key) != fingerprint[key]:
            return "changed"
    return "current"


def write_index_metadata(index_dir, fingerprint, chunk_count):
    """Save a small JSON file in the index folder that records what the index was built from."""
    metadata = dict(fingerprint)
    metadata["source_file"] = KB_PATH
    metadata["chunk_count"] = chunk_count
    metadata["created_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    (Path(index_dir) / "index_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

In [ ]:
fingerprint = make_fingerprint()
index_state = check_index(INDEX_DIR, fingerprint)   # 'missing', 'changed' or 'current'
print("Saved index state:", index_state)

if REBUILD_INDEX or index_state != "current":
    # Expensive step: every chunk is sent to the embedding model
    print("Building a new FAISS index. Every chunk is being embedded...")
    vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)
    built_now = True
else:
    # Cheap step: the vectors are read from disk and no document embeddings are created.
    # Only load index folders that this notebook created (see the security note in Section 8).
    print("Reusing the saved index. No document embeddings will be generated.")
    vector_store = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
    built_now = False

# Vector store information
print("\nVectors in the index:", vector_store.index.ntotal)
print("Dimensions per vector:", vector_store.index.d)

In [ ]:
# A question written in plain English. It uses different words from the record titles on purpose.
sample_question = "Why is my server processor running at almost full capacity?"

# similarity_search embeds the question and returns the TOP_K closest chunks
top_docs = vector_store.similarity_search(sample_question, k=TOP_K)

print("Question:", sample_question)
print("Top", TOP_K, "retrieved documents:")
for rank, doc in enumerate(top_docs, start=1):
    preview = doc.page_content[:250].replace("\n", " | ")
    print(f"\n#{rank} metadata: {doc.metadata}")
    print(preview, "...")

## Section 7 - Persist the FAISS Index Locally

**Why persist?** The FAISS index lives in memory. When the notebook kernel stops, the index is lost. Saving it to disk means the next run can load the vectors instead of recalculating them with the embedding model.

**What `save_local()` creates.** It writes two files into the `faiss_operations_index/` folder:

| File | Purpose |
|---|---|
| `index.faiss` | The vector index: the vectors for all chunks, arranged so that searches are fast. |
| `index.pkl` | The document store: the text and metadata of each chunk, and the link from each vector to its chunk. |

This notebook adds a third, small file:

| File | Purpose |
|---|---|
| `index_metadata.json` | The checksum of the source CSV, the embedding model, the chunk settings, and the build time. Section 9 uses it to decide whether the index is still valid. |

**What is needed to reload an index?** The folder with these files, and the same embedding model. The vectors on disk only make sense for the model that produced them, which is why the metadata stores the model name.

**Important:** `index.pkl` is a Python pickle file. Only load index files that you created yourself or obtained from a trusted source. Section 8 explains this in detail.

In [ ]:
if built_now:
    # Save the FAISS files, then write the metadata file next to them
    vector_store.save_local(INDEX_DIR)
    write_index_metadata(INDEX_DIR, fingerprint, chunk_count=len(chunks))
    print("Index saved to:", INDEX_DIR)
else:
    print("The index was loaded from disk, so there is nothing new to save.")

# Show the files in the index folder and their sizes
print(f"\nContents of {INDEX_DIR}/")
for path in sorted(Path(INDEX_DIR).iterdir()):
    print(f"  {path.name:24} {path.stat().st_size:>10,} bytes")

print("\n--- index_metadata.json ---")
print((Path(INDEX_DIR) / "index_metadata.json").read_text(encoding="utf-8"))

## Section 8 - Reload an Existing FAISS Index

**Why reload?** Reloading reads the saved files and rebuilds the in-memory index. The document vectors are read from disk, so **no document embeddings are generated**. A search still needs one embedding call to embed the question.

**The loading call:**

```python
FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
```

The `embeddings` object is needed so that FAISS can embed new questions with the same model that created the index.

**Security warning: `allow_dangerous_deserialization=True`**

- `index.pkl` is a Python pickle file. Loading a pickle can run any code it contains.
- LangChain refuses to read this file unless you pass `allow_dangerous_deserialization=True`. The flag is an explicit acknowledgement of this risk.
- Only load index folders that you created yourself or that come from a trusted source. Never download or load an index file from an unknown person or website.
- Do not share your index folder with untrusted people. Treat it like a program.

The reload cell below and the Section 6 cell use the flag only for the index created by this notebook.

In [ ]:
index_folder = Path(INDEX_DIR)

# Step 1: check that the saved index exists
if not index_folder.exists():
    raise FileNotFoundError("No saved index found. Run Sections 6 and 7 first.")

# Step 2: load the saved index. No document embeddings are generated.
loaded_vector_store = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,   # safe here: this notebook created the index
)
print("Index loaded from disk. Vectors in loaded index:", loaded_vector_store.index.ntotal)

# Step 3: compare the results of the in-memory store and the reloaded store.
# The question is embedded ONCE, so both stores receive exactly the same query vector.
# (Embedding the same text twice can give slightly different numbers, so we do not do that here.)
test_question = "What should be checked when server CPU utilization exceeds 90%?"
test_vector = embeddings.embed_query(test_question)
original_results = vector_store.similarity_search_with_score_by_vector(test_vector, k=TOP_K)
loaded_results = loaded_vector_store.similarity_search_with_score_by_vector(test_vector, k=TOP_K)

# Compare the text, metadata and score of each result, rank by rank
same_results = all(
    original_doc.page_content == loaded_doc.page_content
    and original_doc.metadata == loaded_doc.metadata
    and round(original_score, 6) == round(loaded_score, 6)
    for (original_doc, original_score), (loaded_doc, loaded_score) in zip(original_results, loaded_results)
)
print("\nResults from the original store and the reloaded store are identical:", same_results)

for rank, (doc, score) in enumerate(loaded_results, start=1):
    print(f"#{rank} distance={score:.4f} metadata={doc.metadata}")

## Section 9 - Index Reuse Workflow

Every run of this notebook follows the same decision. Here is the complete rule:

| Saved index state | What happens | Document embedding calls |
|---|---|---|
| `missing` (no index on disk) | Build from the chunks, then save | Yes, one per chunk |
| `changed` (CSV, chunk settings or embedding model differ) | Rebuild, then save | Yes, one per chunk |
| `current` and `REBUILD_INDEX = False` | Load from disk | **No** |
| Any state with `REBUILD_INDEX = True` | Rebuild, then save | Yes, one per chunk |

**Why a checksum?** A SHA-256 checksum is a fingerprint of a file's bytes. Changing even one character produces a different checksum. The notebook stores the checksum when the index is built and compares it on every run.

**Why not the file's modified date?** Copying or re-saving a file changes its timestamp even when the content is identical. A checksum changes only when the content changes.

**Restarting the notebook does not trigger a rebuild.** The checksum is read from the file, not from memory, so a restarted kernel still finds the index `current`.

**When to rebuild:** after you change the CSV (add, edit or remove records), change `CHUNK_SIZE`, `CHUNK_OVERLAP` or `EMBEDDING_MODEL`, or set `REBUILD_INDEX = True` for a one-off rebuild.

In [ ]:
# 1. Current state: the saved index matches the current CSV and settings
print("Current index state:", check_index(INDEX_DIR, make_fingerprint()))

# 2. Simulate an edit to the CSV without changing the file.
#    One extra record is added in memory only, and the checksum of the result is calculated.
edited_bytes = Path(KB_PATH).read_bytes() + b"\nOPS-999,Server,Demo record,Demo only,Demo only,Low,Demo team"
edited_fingerprint = make_fingerprint()
edited_fingerprint["source_sha256"] = hashlib.sha256(edited_bytes).hexdigest()

# 3. The checksum no longer matches, so the saved index is reported as 'changed'
print("Index state after a simulated edit:", check_index(INDEX_DIR, edited_fingerprint))

# 4. The rebuild flag overrides everything else
print("\nREBUILD_INDEX is:", REBUILD_INDEX, "(set it to True to force a rebuild on the next run)")

## Section 10 - Create a LangChain Retriever

**What does a retriever do?** A retriever answers one question: "which documents are relevant to this question?" It hides the search details, and any LangChain chain can call it with `retriever.invoke(question)`.

**How is it different from querying the vector store?** The vector store has save, load and search methods. The retriever exposes only the search, in the standard LangChain form. `vector_store.as_retriever()` turns the vector store into a retriever.

**What does top-k control?** `k` is the number of documents returned for each question. A small `k` gives focused context but may miss a useful record. A large `k` gives more context but adds noise and makes the LLM prompt longer. The default in this lab is `TOP_K = 3`.

**How does this support question answering?** The retriever selects the records. The LLM in Section 11 reads only those records and writes the answer.

**Cost note:** retrieval itself does not call the LLM. Each question still needs one embedding call, to embed the question. Each retriever call embeds the question again, so in rare cases two calls can rank near-tied records differently.

In [ ]:
# Retriever built from the FAISS vector store. It returns the top TOP_K documents.
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

question = "How should a backup failure be handled?"
retrieved_docs = retriever.invoke(question)

print("Question:", question)
print("Documents returned with k =", TOP_K, ":", len(retrieved_docs))
for doc in retrieved_docs:
    preview = doc.page_content[:120].replace("\n", " | ")
    print("  row", doc.metadata.get("row"), "|", preview)

# Change the top-k value to 5 and compare. The first three results should normally match the k = 3 list.
retriever_k5 = vector_store.as_retriever(search_kwargs={"k": 5})
retrieved_k5 = retriever_k5.invoke(question)

print("\nRows with k = 3:", [doc.metadata.get("row") for doc in retrieved_docs])
print("Rows with k = 5:", [doc.metadata.get("row") for doc in retrieved_k5])
print("Extra documents at ranks 4 and 5:")
for doc in retrieved_k5[len(retrieved_docs):]:
    preview = doc.page_content[:120].replace("\n", " | ")
    print("  row", doc.metadata.get("row"), "|", preview)

## Section 11 - Connect the Retriever to an LLM

The retriever supplies the context. The LLM writes the answer from that context. The chain works in five steps:

1. The **question** is sent to the retriever, which returns the top `TOP_K` records.
2. The records are joined into one **context** block.
3. The **prompt** combines the context, the question and the rules.
4. The **LLM** (`ChatOpenAI`) writes the answer from the prompt.
5. `StrOutputParser` turns the model's reply into plain text.

**The prompt rules.** The prompt tells the model to use only the retrieved context, not to invent steps, thresholds or contacts, and to reply with a fixed sentence when the answer is not in the knowledge base. Wrong operational steps can cause real damage, so these rules matter.

**Chain syntax.** Each `|` passes the output of one step to the next. This is the same LCEL pattern as Module 7 Lab 1, so it is not explained again here.

In [ ]:
RAG_PROMPT_TEXT = """You are an IT Operations Knowledge Assistant.
Answer the question using ONLY the context below.
Do not invent procedures, steps, thresholds or contacts that are not in the context.
If the context does not contain the answer, reply with exactly this sentence:
"The answer could not be found in the available Operations Knowledge Base."

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEXT)


def format_docs(docs):
    """Join the retrieved chunks into one block of context text."""
    return "\n\n".join(doc.page_content for doc in docs)


# Question -> retriever -> context text -> prompt -> LLM -> plain-text answer
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("RAG chain built. Each question retrieves the top", TOP_K, "chunks.")

## Section 12 - Ask IT Operations Questions

`ask_question()` sends one question through the RAG chain and returns the answer.

The loop below asks five typical operations questions. For each one it first shows the retrieved records (the context the LLM is allowed to use), then the answer. Check that each answer matches its context.

The chain retrieves the records internally. The loop calls the retriever again only to **display** the context, so each question costs one extra embedding call.

The final cell asks a question that is **not** covered by the knowledge base. A vector search always returns the nearest records, even when none of them is relevant. The prompt rules are what stop the LLM from inventing an answer. A correct assistant should reply with the "could not be found" sentence, not with a made-up procedure.

In [ ]:
def ask_question(question):
    """Send one question through the RAG chain and return the answer text."""
    response = rag_chain.invoke(question)
    return response


ops_questions = [
    "What should be checked when server CPU utilization exceeds 90%?",
    "What steps should be followed when a backup fails?",
    "How should network connectivity issues be investigated?",
    "What is the escalation procedure for critical incidents?",
    "What should be done when disk utilization is high?",
]

for number, question in enumerate(ops_questions, start=1):
    print(f"=== Q{number}: {question}")

    # Show the retrieved context (the records the LLM may use)
    for doc in retriever.invoke(question):
        preview = doc.page_content[:90].replace("\n", " | ")
        print("  context: row", doc.metadata.get("row"), "|", preview)

    print(f"A{number}: {ask_question(question)}\n")

In [ ]:
missing_question = "What is the procedure for replacing a faulty UPS battery in the data centre?"

print("Question (not in the knowledge base):", missing_question)

# The nearest records are still retrieved, but none of them describes UPS batteries
for doc in retriever.invoke(missing_question):
    preview = doc.page_content[:90].replace("\n", " | ")
    print("  context: row", doc.metadata.get("row"), "|", preview)

print("\nAnswer:", ask_question(missing_question))

## Section 13 - Index Reuse and API Cost

**First execution (no saved index):**

1. Load the CSV and split it into chunks.
2. Send every chunk to the embedding model (API cost).
3. Create the FAISS index in memory.
4. Save `index.faiss`, `index.pkl` and `index_metadata.json` to disk.

**Later execution (saved index found):**

1. Load the CSV and calculate its checksum.
2. Compare the checksum and settings with `index_metadata.json`.
3. If they match, load the saved index. **No document embeddings are generated.**
4. Create the retriever and ask questions.

**What still costs API calls on every run:**

- Each question is embedded once, for the similarity search. This cost is small.
- The LLM reads the retrieved context and writes the answer. This is the main cost per question.

**What persistence does and does not change:**

- It avoids embedding all chunks again. For a knowledge base that changes rarely, this is the largest saving.
- It does **not** remove all API usage. Questions still need embeddings and an LLM answer.
- It does **not** make a rebuild cheaper. Any change to the CSV, the chunk settings or the embedding model requires embedding all chunks again. Old vectors cannot be mixed with vectors from a different model.

## Section 14 - Hands-On Exercises and Quiz

### Exercises

**Exercise 1 - Create a Vector Store.** Delete the `faiss_operations_index/` folder, then run the notebook from the top. Note how many chunks were embedded and how many vectors the index contains.

**Exercise 2 - Persist the Index.** After Section 7, open the `faiss_operations_index/` folder in the VS Code Explorer. Open `index_metadata.json` and find the checksum, the embedding model and the chunk count.

**Exercise 3 - Reload the Index.** Restart the notebook kernel (use the Restart button on the notebook toolbar). Run the notebook again from the top. Confirm that the state is `current` and that no document embeddings are generated.

**Exercise 4 - Change Retrieval Top-K.** Change `TOP_K` in the configuration cell from 3 to 5. In the same kernel session, re-run the notebook from Section 10 onwards. Compare the retrieved records and the answers. Changing `TOP_K` does not change the index, so the index is reused.

**Exercise 5 - Rebuild after Dataset Changes.** Open `operations_knowledge_base.csv` and add two new records with the same seven columns, for example `OPS-030` and `OPS-031`. Put any text that contains a comma in double quotes. Save the file and run the notebook from the top. The state should be `changed`, the index should be rebuilt, and the new records should be retrievable. To return to the original data, remove the new rows and run the notebook again.

### Quiz

**1. Vector stores.** What is the main job of a vector store in a RAG application?

- A. Generate the final answer with an LLM
- B. Store chunk vectors and find the vectors closest to a query vector
- C. Split documents into chunks
- D. Store the OpenAI API key securely

**2. Embeddings.** Two sentences use different words but have the same meaning. What do you expect from their embeddings?

- A. Identical vectors
- B. Random, unrelated vectors
- C. Vectors that point in similar directions and sit close together
- D. Vectors of different lengths

**3. FAISS persistence.** What does `save_local()` store?

- A. Only the OpenAI API key
- B. The vector index and the document store with each chunk's text and metadata
- C. The LLM's answers to earlier questions
- D. A copy of the original CSV file only

**4. Index reloading.** You restart the notebook, and the CSV and settings have not changed. What should happen?

- A. The index is rebuilt and every chunk is embedded again
- B. The saved index is loaded and no document embeddings are generated
- C. The saved index is deleted
- D. The chunk size is changed automatically

**5. LangChain retrievers.** What does `top-k` control in a retriever?

- A. The number of documents returned for each query
- B. The size of each chunk in characters
- C. The number of tokens in the LLM answer
- D. The number of CSV files that are loaded

---

## Instructor Section - Quiz Answers and Notes

*For the trainer. Remove this section before sharing the notebook with participants.*

**Quiz answers**

1. **B.** A vector store keeps the vectors and finds the nearest ones. The LLM generates the answer (A), the chunking is done by the splitter (C), and the API key belongs in `.env` (D).
2. **C.** Embedding models place texts with similar meaning close together, even with different words. Identical vectors (A) would mean the texts are indistinguishable, and random vectors (B) would make search useless.
3. **B.** `save_local()` writes `index.faiss` (vectors) and `index.pkl` (document store with text and metadata). It never stores the API key (A), LLM answers (C) or the original CSV (D).
4. **B.** Reloading reads the saved files, so no document embeddings are generated. Rebuilding (A) is needed only when the source or settings change. Deleting the index (C) and changing the chunk size (D) are not automatic.
5. **A.** `k` controls how many documents the retriever returns. Chunk size (B) is set in the splitter, the answer length (C) is controlled by the LLM, and the number of CSV files (D) is set by the loader.

**Notes for the trainer**

- **Expected states during the exercises:** the first run reports `missing`, a second run after a restart reports `current`, and a run after editing the CSV reports `changed`. The embedding count should be zero on any run that reports `current`.
- **Answer wording:** LLM output can vary slightly between runs even with `temperature=0`. Judge answers by whether they stay inside the retrieved context and whether the out-of-scope question gets the "could not be found" sentence.
- **Retrieval quality:** the question "Why is my server processor running at almost full capacity?" should retrieve CPU-related records near the top. Use it to explain why semantic search finds records that keyword search would miss.
- **Security discussion:** use Section 8 to stress that pickle files are code. Participants should never load an index folder from an unknown source.
- **Cost discussion:** point out that questions still cost embedding and LLM calls. Only the document-embedding cost is avoided by reuse.